# Semi-synthetic results

This notebook reports the completed multi-dataset CoxPH-margin experiment with Gaussian, Clayton, Frank, and Gumbel copulas from `results/semi-synthetic`. It loads the cross-dataset aggregate when available, otherwise combines the completed per-dataset result files. The semi-synthetic cohorts retain each source dataset's original censoring rate.

In [ ]:
from pathlib import Path
import os
import re
import shutil
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")
plt.rcParams.update({
    "axes.labelsize": "large",
    "axes.titlesize": "large",
    "font.size": 16.0,
    "legend.fontsize": "large",
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsfonts} \usepackage{amstext} \usepackage{bm}",
    "savefig.bbox": "tight",
})

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
CONFIG_PATH = ROOT / "configs" / "semi_synthetic.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
RESULT_CANDIDATES = [ROOT / "results" / "semi-synthetic"]
FIGURE_DIR = ROOT / "paper" / "figures"
TABLE_DIR = ROOT / "paper" / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
RANK_TAUS = (0.0, 0.5)

In [ ]:
def snake_case(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def canonicalize(frame):
    out = frame.rename(columns={column: snake_case(column) for column in frame.columns}).copy()
    aliases = {
        "dataset_name": "dataset", "target_censoring": "target_censoring_rate",
        "censoring_rate": "target_censoring_rate", "kendall_tau": "target_kendall_tau",
        "oracle_joint_ise": "oracle_joint_survival_ise",
    }
    out = out.rename(columns={key: value for key, value in aliases.items() if key in out and value not in out})
    if "model" in out:
        names = {"dvfm": "DVFM", "coxph": "CoxPH", "deepsurv": "DeepSurv", "rsf": "RSF",
                 "gbsa": "GBSA", "mtlr": "MTLR", "deephit": "DeepHit", "hacsurv": "HACSurv",
                 "hacsurv_2d": "HACSurv", "clayton_aft": "ClaytonAFT",
                 "bayesian_cox_gamma_frailty": "BayesianCoxGammaFrailty", "bayesiancoxgammafrailty": "BayesianCoxGammaFrailty"}
        out["model"] = out["model"].astype(str).str.lower().map(names).fillna(out["model"].astype(str))
        if "latent_dim" in out:
            latent_dim = pd.to_numeric(out["latent_dim"], errors="coerce")
            nonprimary = out["model"].eq("DVFM") & latent_dim.ne(1) & latent_dim.notna()
            out.loc[nonprimary, "model"] = latent_dim.loc[nonprimary].map(lambda value: f"DVFM-z{int(value)}")
    if "copula" in out:
        out["copula"] = out["copula"].astype(str).str.title()
    return out


result_path = next((path / "results_raw.csv" for path in RESULT_CANDIDATES if (path / "results_raw.csv").exists()), None)
if result_path is not None:
    results = canonicalize(pd.read_csv(result_path))
    DATA_SOURCE = "REAL RESULTS (cross-dataset aggregate)"
else:
    dataset_result_paths = [
        result for root in RESULT_CANDIDATES if root.exists()
        for result in sorted(root.glob("*/results_raw.csv"))
    ]
    if not dataset_result_paths:
        searched = ", ".join(str(path.relative_to(ROOT)) for path in RESULT_CANDIDATES)
        raise FileNotFoundError(f"No semi-synthetic results found in: {searched}")
    results = canonicalize(pd.concat((pd.read_csv(path) for path in dataset_result_paths), ignore_index=True))
    DATA_SOURCE = f"REAL RESULTS ({len(dataset_result_paths)} completed dataset jobs)"

required = {"dataset", "copula", "target_kendall_tau", "target_censoring_rate", "repeat", "model", "oracle_ibs"}
missing = required - set(results.columns)
if missing:
    raise ValueError(f"Results are missing canonical columns: {sorted(missing)}")

CONFIG_TAU_VALUES = [float(value) for value in config["data"]["kendall_tau"]]
CONFIG_POSITIVE_COPULAS = {name.title() for name in config["data"]["copulas"]}
tau = pd.to_numeric(results["target_kendall_tau"], errors="coerce")
configured_tau = np.isclose(tau.to_numpy()[:, None], np.asarray(CONFIG_TAU_VALUES)).any(axis=1)
configured_copula = (np.isclose(tau, 0.0) & results["copula"].eq("Independence")) \
| (~np.isclose(tau, 0.0) & results["copula"].isin(CONFIG_POSITIVE_COPULAS))
results = results.loc[configured_tau & configured_copula].copy()
if results.empty:
    raise ValueError("No rows match the active semi_synthetic.yaml condition grid.")

# Table 1 is derived from completed runs rather than a hand-written table.
dataset_specs = {item["name"]: item for item in config["data"]["datasets"]}
characteristics = (
    results.groupby("dataset", as_index=False)
    .agg(**{"$N$": ("num_samples", "median"),
            "Encoded features": ("num_features", "median"),
            "Original censoring rate": ("target_censoring_rate", "median")})
    .rename(columns={"dataset": "Dataset"})
)
characteristics["Raw features"] = characteristics["Dataset"].map(
    lambda name: len(dataset_specs[name].get("numeric_features", [])) + len(dataset_specs[name].get("categorical_features", []))
)
characteristics["Split"] = "70/10/20"
characteristics = characteristics[["Dataset", "$N$", "Raw features", "Encoded features", "Original censoring rate", "Split"]]
for column in ["$N$", "Raw features", "Encoded features"]:
    characteristics[column] = characteristics[column].round().astype(int)

print(DATA_SOURCE, f"| rows={len(results):,} | datasets={results['dataset'].nunique()}")

REAL RESULTS (cross-dataset aggregate) | rows=15,600 | datasets=12


## Supplementary shared-latent recovery hexbin plot

Hexbin density makes the held-out subject-level recovery relationship legible without overplotting. The four panels use the real SUPPORT semi-synthetic exports at target $\tau=0.50$, pooling standardized held-out subject-run observations across the ten repeats for the Gaussian, Clayton, Gumbel, and Frank copulas. Each repeat retains its own validation-derived calibration; no test subjects are used to fit an affine mapping. Frank's generating latent is discrete, so tied structure is expected in that panel.

In [ ]:
HEXBIN_DATASET = "support"
HEXBIN_TAU = 0.50
HEXBIN_COPULAS = {
    "Gaussian": "gaussian",
    "Clayton": "clayton",
    "Gumbel": "gumbel",
    "Frank": "frank",
}

def latent_run_paths(dataset, copula, tau=HEXBIN_TAU):
    token = f"_{copula}_tau_{tau:g}_"
    matches = sorted(
        path for root in RESULT_CANDIDATES if (root / dataset).exists()
        for path in (root / dataset).rglob("latent_recovery_test.csv")
        if token in path.parent.name.lower()
    )
    # Optionally warn rather than hard-fail when some seeds are missing
    if not matches:
        raise FileNotFoundError(
            f"No latent exports found for {dataset}/{copula} at tau={tau:g}."
        )
    if len(matches) != len(config["seeds"]):
        import warnings
        warnings.warn(
            f"Expected {len(config['seeds'])} {dataset}/{copula} seeds at tau={tau:g}; "
            f"found {len(matches)} (some may be unavailable_no_true_frailty)."
        )
    return matches

hexbin_runs = {label: latent_run_paths(HEXBIN_DATASET, copula) for label, copula in HEXBIN_COPULAS.items()}

hexbin_subjects = {
    label: pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
    for label, paths in hexbin_runs.items()
}

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.5), constrained_layout=True)
axes = axes.ravel()
images = []
for ax, (label, part) in zip(axes, hexbin_subjects.items()):
    required_columns = {"true_z", "learned_z_calibrated"}
    missing_columns = required_columns - set(part.columns)
    if missing_columns:
        raise ValueError(f"SUPPORT latent data are missing columns: {sorted(missing_columns)}")
    image = ax.hexbin(part["true_z"], part["learned_z_calibrated"], gridsize=42,
                      mincnt=1, cmap="viridis", linewidths=0)
    images.append(image)
    limits = np.nanpercentile(np.r_[part["true_z"], part["learned_z_calibrated"]], [0.5, 99.5])
    ax.plot(limits, limits, "--", color="black", linewidth=1, label="Ideal")
    correlation = part[["true_z", "learned_z_calibrated"]].corr().iloc[0, 1]
    ax.set(title=f"{label} copula\nPearson r = {correlation:.3f}",
           xlabel="True shared latent z", ylabel="Recovered shared latent z")
    ax.legend(frameon=False)
common_vmax = max(float(image.get_array().max()) for image in images)
for image in images:
    image.set_clim(1, common_vmax)
fig.colorbar(images[-1], ax=axes, label="Subjects per hexagon", shrink=0.8)
fig.suptitle(rf"SUPPORT frailty recovery ($\tau={HEXBIN_TAU:.2f}$)", fontsize=16)
out = FIGURE_DIR / "semi_synthetic_frailty_recovery_hexbin.pdf"
fig.savefig(out)
plt.show()
print("SUPPORT held-out subject-runs:", {label: len(part) for label, part in hexbin_subjects.items()})
print(f"wrote {out}")

## Figure 1: model ranks across semi-synthetic datasets

Models are compared separately at target Kendall's $\tau=0$ and $\tau=0.5$. The independence condition supplies the $\tau=0$ row. At $\tau=0.5$, ranks are computed within each paired dataset/copula/seed/metric cell and then averaged equally over seeds and the Gaussian, Clayton, Frank, and Gumbel copulas within each dataset. A failed model receives one rank below the worst successful model in its paired cell. Rank 1 is best. Points show median dataset ranks; horizontal bars are 95% non-parametric bootstrap intervals obtained by resampling datasets.

In [ ]:
RANK_METRICS = {
    "oracle_ci": ("Oracle CI", True),
    "oracle_ibs": ("Oracle IBS", False),
    "oracle_mae": ("Oracle MAE", False),
}
N_BOOTSTRAPS = 20_000
BOOTSTRAP_SEED = 2026
MODEL_LABELS = {
    "DVFM": "DVFM ($z=1$)", "DVFM-z0": "DVFM ($z=0$)",
    "BayesianCoxGammaFrailty": "Cox-Gamma Frailty",
    "RSF": "RSF",
    "GBSA": "Gradient Boosting",
    "ClaytonAFT": "Clayton AFT", "HACSurv": "HACSurv",
}

SEED_KEYS = ["dataset", "copula", "target_kendall_tau", "target_censoring_rate", "repeat"]
EXPECTED_REPEATS = len(config["seeds"])

def seed_level_metric_ranks(frame, metric, higher_is_better):
    """Rank paired runs, penalizing only the atomic seeds that failed."""
    work = frame[SEED_KEYS + ["model", metric, "numerical_failure"]].copy()
    if work.duplicated(SEED_KEYS + ["model"]).any():
        raise ValueError(f"Duplicate model rows found while ranking {metric}.")
    values = pd.to_numeric(work[metric], errors="coerce")
    failure = work["numerical_failure"].fillna(False)
    if failure.dtype != bool:
        failure = failure.astype(str).str.lower().eq("true")
    valid = np.isfinite(values) & ~failure
    work["_score"] = values.where(valid)
    work["seed_rank"] = work.groupby(SEED_KEYS)["_score"].rank(
        ascending=not higher_is_better, method="min"
    )
    worst_completed = work.groupby(SEED_KEYS)["seed_rank"].transform("max")
    if worst_completed[~valid].isna().any():
        raise ValueError(f"A {metric} seed cell has no successful model to define a failure rank.")
    work.loc[~valid, "seed_rank"] = worst_completed.loc[~valid] + 1
    work["failed"] = ~valid
    return work

# Rank within paired seeds first. For tau=0.5, average the four
# copula-specific ranks equally within each dataset before bootstrapping datasets.
rank_tables_by_tau = {}
for tau_value in RANK_TAUS:
    rank_input = results[np.isclose(results["target_kendall_tau"], tau_value)].copy()
    if rank_input.empty:
        raise ValueError(f"No results for tau={tau_value:g}.")
    expected_copulas = {"Independence"} if np.isclose(tau_value, 0.0) else CONFIG_POSITIVE_COPULAS
    observed_copulas = set(rank_input["copula"].unique())
    if observed_copulas != expected_copulas:
        raise ValueError(
            f"Unexpected copulas for tau={tau_value:g}: expected {sorted(expected_copulas)}, "
            f"found {sorted(observed_copulas)}."
        )
    rank_tables = {}
    for metric, (_, higher_is_better) in RANK_METRICS.items():
        seed_ranks = seed_level_metric_ranks(rank_input, metric, higher_is_better)
        repeat_counts = seed_ranks.groupby(["dataset", "copula", "model"])["repeat"].nunique()
        if not repeat_counts.eq(EXPECTED_REPEATS).all():
            incomplete = repeat_counts[~repeat_counts.eq(EXPECTED_REPEATS)]
            raise ValueError(
                f"Incomplete seed grid for tau={tau_value:g}, {metric}: {incomplete.to_dict()}"
            )
        scenario_ranks = (
            seed_ranks.groupby(["dataset", "copula", "model"], as_index=False)["seed_rank"].mean()
        )
        rank_tables[metric] = (
            scenario_ranks.groupby(["dataset", "model"])["seed_rank"].mean().unstack("model")
        )
    rank_tables_by_tau[tau_value] = rank_tables

all_rank_tables = [
    table for rank_tables in rank_tables_by_tau.values() for table in rank_tables.values()
]
models = sorted(set.intersection(*(set(table.columns) for table in all_rank_tables)))
if not models:
    raise ValueError("No model has all rank metrics at both tau levels.")
order_components = [table.reindex(columns=models).median(axis=0) for table in all_rank_tables]
model_order = pd.concat(order_components, axis=1).mean(axis=1).sort_values().index.tolist()

rng = np.random.default_rng(BOOTSTRAP_SEED)
rank_intervals = {}
for tau_value, rank_tables in rank_tables_by_tau.items():
    rank_intervals[tau_value] = {}
    for metric, ranks in rank_tables.items():
        values = ranks.reindex(columns=model_order).to_numpy(dtype=float)
        sampled_rows = rng.integers(0, len(values), size=(N_BOOTSTRAPS, len(values)))
        bootstrap_medians = np.median(values[sampled_rows], axis=1)
        rank_intervals[tau_value][metric] = pd.DataFrame(
            {"median": np.median(values, axis=0),
             "low": np.quantile(bootstrap_medians, 0.025, axis=0),
             "high": np.quantile(bootstrap_medians, 0.975, axis=0)},
            index=model_order,
        )

fig, axes = plt.subplots(2, 3, figsize=(14.5, 9.0), sharex=True, sharey=True, constrained_layout=True)
positions = np.arange(len(model_order))
for row_index, tau_value in enumerate(RANK_TAUS):
    for axis, (metric, (title, _)) in zip(axes[row_index], RANK_METRICS.items()):
        interval = rank_intervals[tau_value][metric].loc[model_order]
        for y_pos, (model, row) in enumerate(interval.iterrows()):
            is_dvfm = model.startswith("DVFM")
            axis.errorbar(
                row["median"], y_pos,
                xerr=[[row["median"] - row["low"]], [row["high"] - row["median"]]],
                fmt="*" if is_dvfm else "o", markersize=10 if is_dvfm else 5.5,
                capsize=2.5, color="#0072B2" if is_dvfm else "#4D4D4D",
            )
        axis.set(
            title=rf"{title} ($\tau={tau_value:g}$)",
            xlabel="Rank (better to the left)",
            xlim=(0.5, len(model_order) + 0.5),
        )
        axis.set_xticks(np.arange(1, len(model_order) + 1))
        axis.grid(axis="x", color="0.9", linewidth=0.8)
    axes[row_index, 0].set(
        yticks=positions,
        yticklabels=[MODEL_LABELS.get(model, model) for model in model_order],
        ylabel="Model",
    )
axes[0, 0].invert_yaxis()
fig.suptitle("Median model rank across datasets by dependence strength", fontsize=14)
out = FIGURE_DIR / "semi_synthetic_model_ranks.pdf"
fig.savefig(out, bbox_inches="tight")
plt.show()
print(
    f"{DATA_SOURCE} | {results['dataset'].nunique()} datasets; "
    f"tau levels={RANK_TAUS}; {N_BOOTSTRAPS:,} dataset bootstraps | wrote {out}"
)

## Figure 2: recovery and mechanistic prediction benefit

The horizontal axis is target Kendall's $\tau$, the active experimental factor. For each dataset, metrics are averaged over seeds before pooling; intervals use datasetsâ€”not pooled subjects or seedsâ€”as the unit of uncertainty. Each dataset retains its original censoring rate, which is deliberately not used as a plotting stratum.

In [ ]:
def dataset_balanced_summary(frame, metric, sigma_level=1.0):
    # Average seeds within each dataset, then quantify cross-dataset variation.
    per_dataset = frame.groupby(["dataset", "copula", "target_kendall_tau"], as_index=False)[metric].mean()
    out = per_dataset.groupby(["copula", "target_kendall_tau"])[metric].agg(["mean", "std", "count"]).reset_index()
    out["error"] = sigma_level * out["std"].fillna(0)
    return out

dvfm = results[results["model"].eq("DVFM")].copy()
mechanistic = results[results["model"].isin(["DVFM", "DVFM-z0"])].pivot_table(
    index=["dataset", "copula", "target_kendall_tau", "repeat"], columns="model", values="oracle_ibs"
).dropna().reset_index()
mechanistic["oracle_ibs_gain"] = mechanistic["DVFM-z0"] - mechanistic["DVFM"]
summaries = [
    (dataset_balanced_summary(dvfm, "frailty_spearman"), "Shared-latent Spearman correlation", "A  Individual shared-latent recovery"),
    (dataset_balanced_summary(dvfm, "absolute_conditional_kendall_tau_error"), r"Absolute Kendall's $\tau$ error", "B  Dependence calibration"),
    (dataset_balanced_summary(dvfm, "oracle_joint_survival_ise"), "Joint-survival ISE", "C  Joint-distribution recovery"),
    (dataset_balanced_summary(mechanistic, "oracle_ibs_gain"), r"$\mathrm{IBS}_{z=0}-\mathrm{IBS}_{z=1}$", "D  Predictive benefit of the shared latent"),
]
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
copula_order = ["Independence"] + [name.title() for name in config["data"]["copulas"]]
palette = dict(zip(copula_order, default_colors[:len(copula_order)]))
tau_levels = CONFIG_TAU_VALUES
fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.0), sharex=False, constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.02, h_pad=0.02, wspace=0.02, hspace=0.02)
for ax, (summary, ylabel, title) in zip(axes.flat, summaries):
    for copula, part in summary.groupby("copula"):
        part = part.sort_values("target_kendall_tau")
        color = palette.get(copula, default_colors[0])
        ax.fill_between(part["target_kendall_tau"], part["mean"] - part["error"], part["mean"] + part["error"], color=color, alpha=0.15, linewidth=0)
        ax.plot(part["target_kendall_tau"], part["mean"], marker="o", label=copula, color=color)
    ax.set(title=title, xlabel=r"Target Kendall's $\tau$", ylabel=ylabel)
    ax.set_xticks(tau_levels, [f"{value:g}" for value in tau_levels])
axes[0, 0].set_ylim(-0.05, 1.0)
axes[0, 0].legend(title="Copula", loc="lower right", frameon=False)
axes[1, 1].axhline(0, color="0.35", linestyle="--", linewidth=1)
for ax in axes.flat:
    ax.grid(True, alpha=0.5)
out = FIGURE_DIR / "semi_synthetic_recovery.pdf"
fig.savefig(out)
plt.show()
print(f"{DATA_SOURCE} | wrote {out}")

## Figure 3: subject-level shared-latent recovery

These figures use one held-out completed semi-synthetic run. Set `FRAILTY_RUN_CONTAINS` to select a particular latent-recovery directory; otherwise the first completed export is used. The fitted latent-scale mapping is learned on that run's validation split only. The cross-family estimand is the generating shared latent; for the selected Clayton run this is standardized log-Gamma frailty.

In [ ]:
FRAILTY_RUN_CONTAINS = "support_clayton_tau_0.5"  # Case-insensitive latent_recovery run-directory substring.

latent_paths = sorted(
    path for root in RESULT_CANDIDATES if root.exists()
    for path in root.rglob("latent_recovery_test.csv")
)
if FRAILTY_RUN_CONTAINS:
    latent_paths = [path for path in latent_paths if FRAILTY_RUN_CONTAINS.lower() in str(path.parent).lower()]
    if not latent_paths:
        raise FileNotFoundError(f"No latent-recovery run matches {FRAILTY_RUN_CONTAINS!r}.")
if not latent_paths:
    raise FileNotFoundError("No held-out semi-synthetic latent-recovery exports were found.")

frailty_test = pd.read_csv(latent_paths[0])
FRAILTY_SOURCE = f"held-out subjects: {latent_paths[0].parent.name}"
required_latent_columns = {"event", "true_z", "learned_mu_aligned", "learned_z_calibrated"}
missing_latent_columns = required_latent_columns.difference(frailty_test.columns)
if missing_latent_columns:
    raise ValueError(f"Latent-recovery data are missing: {sorted(missing_latent_columns)}")
print(f"Subject-level shared-latent source: {FRAILTY_SOURCE} | n={len(frailty_test):,}")

In [ ]:
def central_limits(frame, x_col, y_col, lower_q=0.01, upper_q=0.99):
    x_low, x_high = frame[x_col].quantile([lower_q, upper_q])
    y_low, y_high = frame[y_col].quantile([lower_q, upper_q])
    lower, upper = float(min(x_low, y_low)), float(max(x_high, y_high))
    padding = 0.04 * (upper - lower)
    return lower - padding, upper + padding

limit_low, limit_high = central_limits(frailty_test, "true_z", "learned_z_calibrated")
panels = [
    ("All test subjects", np.ones(len(frailty_test), dtype=bool)),
    ("Event observed", frailty_test["event"].to_numpy() == 1),
    ("Censored", frailty_test["event"].to_numpy() == 0),
]
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.8), sharex=True, sharey=True)
for ax, (title, mask) in zip(axes, panels):
    subset = frailty_test.loc[mask]
    x = subset["true_z"].to_numpy()
    y = subset["learned_z_calibrated"].to_numpy()
    pearson = float(pd.Series(x).corr(pd.Series(y), method="pearson"))
    spearman = float(pd.Series(x).corr(pd.Series(y), method="spearman"))
    r2 = float(1 - np.sum((y - x) ** 2) / np.sum((x - x.mean()) ** 2))
    ax.scatter(x, y, alpha=0.45, s=18)
    ax.plot(
        [limit_low, limit_high], [limit_low, limit_high],
        linestyle="--", linewidth=1.5, color="#ff7f0e",
        label=r"Perfect recovery: $\hat{z}_i = z_i$",
    )
    ax.text(
        0.04, 0.96,
        f"n = {len(subset):,}\nPearson = {pearson:.2f}\nSpearman = {spearman:.2f}\n$R^2$ = {r2:.2f}",
        transform=ax.transAxes, va="top",
        bbox={"boxstyle": "round", "facecolor": "white", "edgecolor": "0.2", "alpha": 0.9},
    )
    ax.set(xlim=(limit_low, limit_high), ylim=(limit_low, limit_high), title=title)
    ax.grid(alpha=0.18)
    ax.set_xlabel("True standardized log-frailty")
    ax.legend(loc="lower right", frameon=True)
axes[0].set_ylabel("Calibrated DVFM latent")
fig.suptitle("DVFM recovers subject-level frailty, including among censored subjects", y=1.03)
fig.tight_layout()
out = FIGURE_DIR / "semi_synthetic_subject_frailty_recovery.pdf"
fig.savefig(out)
plt.show()
print(f"wrote {out}")


In [ ]:
x = frailty_test["true_z"].to_numpy()
y = frailty_test["learned_mu_aligned"].to_numpy()
x_low, x_high = np.quantile(x, [0.01, 0.99])
y_low, y_high = np.quantile(y, [0.01, 0.99])
slope, intercept = np.polyfit(x, y, 1)
grid = np.linspace(x_low, x_high, 200)
fig, ax = plt.subplots(figsize=(7.2, 5.8))
ax.scatter(x, y, alpha=0.42, s=18)
ax.plot(grid, intercept + slope * grid, linewidth=2)
ax.set(xlim=(x_low, x_high), ylim=(y_low, y_high),
       xlabel="True standardized log-frailty",
       ylabel="DVFM posterior mean, sign-aligned",
       title="Uncalibrated subject-level frailty association")
ax.grid(alpha=0.18)
out = FIGURE_DIR / "semi_synthetic_subject_frailty_uncalibrated.pdf"
fig.savefig(out)
plt.show()
print(f"wrote {out}")


In [ ]:
deciles = frailty_test.copy()
deciles["true_z_decile"] = pd.qcut(deciles["true_z"], q=10, duplicates="drop")
binned = deciles.groupby("true_z_decile", observed=True).agg(
    true_z_mean=("true_z", "mean"),
    calibrated_mean=("learned_z_calibrated", "mean"),
    calibrated_sd=("learned_z_calibrated", "std"),
    n=("true_z", "size"),
).reset_index(drop=True)
binned["sem"] = binned["calibrated_sd"] / np.sqrt(binned["n"])
lower = float(min(binned["true_z_mean"].min(), binned["calibrated_mean"].min()))
upper = float(max(binned["true_z_mean"].max(), binned["calibrated_mean"].max()))
padding = 0.05 * (upper - lower)
fig, ax = plt.subplots(figsize=(7.0, 5.8))
ax.errorbar(binned["true_z_mean"], binned["calibrated_mean"], yerr=1.96 * binned["sem"],
            fmt="o", capsize=3, label="DVFM mean per frailty decile (95% CI)")
ax.plot([lower - padding, upper + padding], [lower - padding, upper + padding], "--", color="0.25",
        label=r"Perfect agreement: $\hat{z}=z$")
ax.set(xlim=(lower - padding, upper + padding), ylim=(lower - padding, upper + padding),
       xlabel="Average true frailty in each subject group",
       ylabel="Average DVFM-predicted frailty",
       title="Prediction calibration across frailty deciles")
ax.grid(alpha=0.18)
ax.legend(frameon=False)
fig.tight_layout()
out = FIGURE_DIR / "semi_synthetic_subject_frailty_deciles.pdf"
fig.savefig(out)
plt.show()
print(f"wrote {out}")


## Tables

The notebook writes booktabs-compatible LaTeX directly. Table 1 describes the datasets. Table 2 reports scenario-balanced ranks and paired wins/ties/losses; full dataset-by-model values belong in the supplement.

In [ ]:
if characteristics is None:
    print("dataset_characteristics.csv is absent; write this file during generation to enable Table 1.")
else:
    dataset_table = characteristics.copy()
    for column in dataset_table.select_dtypes(include="number"):
        if "rate" in column.lower():
            dataset_table[column] = dataset_table[column].map(lambda value: f"{value:.2f}")
    latex = dataset_table.to_latex(
        index=False, escape=False,
        caption="Characteristics of the source datasets used for semi-synthetic evaluation.",
        label="tab:semi_synthetic_datasets",
        column_format="l" + "r" * (len(dataset_table.columns) - 1),
    )
    out = TABLE_DIR / "semi_synthetic_datasets.tex"
    out.write_text(latex, encoding="utf-8")
    print(latex)
    print(f"{DATA_SOURCE} | wrote {out}")

In [ ]:
comparison = results[~results["model"].eq("DVFM-z0")].copy()
scenario_keys = ["dataset", "copula", "target_kendall_tau", "target_censoring_rate"]
rank_table = None
for metric, label in [("oracle_ibs", "IBS"), ("oracle_ci", "CI"), ("oracle_mae", "MAE")]:
    higher_is_better = metric == "oracle_ci"
    seed_ranks = seed_level_metric_ranks(comparison, metric, higher_is_better)
    repeat_counts = seed_ranks.groupby(scenario_keys + ["model"])["repeat"].nunique()
    if not repeat_counts.eq(EXPECTED_REPEATS).all():
        incomplete = repeat_counts[~repeat_counts.eq(EXPECTED_REPEATS)]
        raise ValueError(f"Incomplete seed grid for {metric}: {incomplete.to_dict()}")
    scenario_ranks = seed_ranks.groupby(scenario_keys + ["model"], as_index=False)["seed_rank"].mean()
    summary = scenario_ranks.groupby("model", as_index=False)["seed_rank"].mean().rename(columns={"seed_rank": label})
    rank_table = summary if rank_table is None else rank_table.merge(summary, on="model", how="outer")
failure_counts = comparison.groupby("model")["numerical_failure"].sum().rename("Failures")
rank_table = rank_table.merge(failure_counts, left_on="model", right_index=True, how="left")

# W/T/L is an effect-size summary, so it uses only seeds with valid IBS for
# both methods. Failure-aware ranks above retain and penalize failed seeds.
dvfm_ibs = comparison[comparison["model"].eq("DVFM")][SEED_KEYS + ["oracle_ibs", "numerical_failure"]].rename(
    columns={"oracle_ibs": "dvfm_ibs", "numerical_failure": "dvfm_failure"}
)
paired_models = comparison.merge(dvfm_ibs, on=SEED_KEYS, how="left")
valid_pairs = (
    np.isfinite(pd.to_numeric(paired_models["oracle_ibs"], errors="coerce"))
    & np.isfinite(pd.to_numeric(paired_models["dvfm_ibs"], errors="coerce"))
    & ~paired_models["numerical_failure"].fillna(False).astype(bool)
    & ~paired_models["dvfm_failure"].fillna(False).astype(bool)
)
paired_models = paired_models.loc[valid_pairs].copy()
tol = 1e-4
paired_models["outcome"] = np.where(paired_models["oracle_ibs"] < paired_models["dvfm_ibs"] - tol, "W", np.where(paired_models["oracle_ibs"] > paired_models["dvfm_ibs"] + tol, "L", "T"))
wtl = paired_models.groupby(["model", "outcome"]).size().unstack(fill_value=0).reindex(columns=["W", "T", "L"], fill_value=0)
wtl["W/T/L vs DVFM"] = wtl.astype(int).astype(str).agg("/".join, axis=1)
rank_table = rank_table.merge(wtl[["W/T/L vs DVFM"]], left_on="model", right_index=True, how="left")
rank_table.loc[rank_table["model"].eq("DVFM"), "W/T/L vs DVFM"] = "--"
rank_table = rank_table.sort_values("IBS").rename(columns={"model": "Model", "IBS": "IBS rank", "CI": "CI rank", "MAE": "MAE rank"})
for column in ["IBS rank", "CI rank", "MAE rank"]:
    rank_table[column] = rank_table[column].map(lambda value: f"{value:.2f}")
rank_table["Failures"] = rank_table["Failures"].astype(int)
latex = rank_table.to_latex(
    index=False, escape=False,
    caption="Scenario-balanced semi-synthetic performance. Lower mean seed-level rank is better; failed seeds rank one below the worst successful method in their paired cell. Wins, ties, and losses use valid paired oracle-IBS seeds and are reported from each comparator's perspective.",
    label="tab:semi_synthetic_models",
    column_format="lrrrrr",
)
out = TABLE_DIR / "semi_synthetic_model_summary.tex"
out.write_text(latex, encoding="utf-8")
print(latex)
print(f"{DATA_SOURCE} | wrote {out}")